# HW 2 — Data Preparation (4 Points)

- **UC Merced Land Use** (21 classes of aerial imagery) — in-distribution
- **FIDS30** (30 classes of fruit photos) — out-of-distribution (OOD) test pool

Split UC Merced 70/15/15 (stratified) for training. FIDS30 is kept as a flat OOD pool.

## 1. UC Merced (In-Distribution)

In [ ]:
import shutil
from pathlib import Path
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# TODO: Load the UC Merced dataset (you already did this in HW_1)
UC = load_dataset(___)
class_names = UC["train"].features["label"].names

all_images, all_labels = [], []
for split in UC:
    all_images.extend(UC[split]["image"])
    all_labels.extend(UC[split]["label"])

# TODO: Stratified 70% train / 15% val / 15% test (same trick as HW_1: two splits)
train_imgs, temp_imgs, train_labels, temp_labels = train_test_split(
    all_images, all_labels, test_size=___, stratify=all_labels, random_state=42)
val_imgs, test_imgs, val_labels, test_labels = train_test_split(
    temp_imgs, temp_labels, test_size=___, stratify=temp_labels, random_state=42)

print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

In [ ]:
# Save images to disk as PrepData/UC_Merced/<split>/<class>/*.jpg (provided — just run)
UC_TARGET = Path("PrepData/UC_Merced")
if UC_TARGET.exists():
    shutil.rmtree(UC_TARGET)

for split_name, (imgs, labels) in [("Training", (train_imgs, train_labels)),
                                    ("Validation", (val_imgs, val_labels)),
                                    ("Test", (test_imgs, test_labels))]:
    for idx, (img, label) in enumerate(zip(imgs, labels)):
        dest = UC_TARGET / split_name / class_names[label]
        dest.mkdir(parents=True, exist_ok=True)
        img.convert("RGB").save(dest / f"{idx:04d}.jpg")
    print(f"  UC Merced {split_name}: {len(imgs)} images")

## 2. FIDS30 (OOD)

In [ ]:
import urllib.request
import zipfile
import os

DATA_URL = "https://data.vicos.si/datasets/FIDS30/FIDS30.zip"
ZIP_PATH = Path("FIDS30.zip")
EXTRACT_DIR = Path("data")
FIDS_TARGET = Path("PrepData/FIDS30")

# Download + extract (provided — just run)
if not (EXTRACT_DIR / "FIDS30").exists():
    if not ZIP_PATH.exists():
        print("Downloading FIDS30 dataset...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extracted.")

# Symlink FIDS30 images into PrepData/FIDS30/<class>/*.jpg (provided — just run)
if FIDS_TARGET.exists():
    shutil.rmtree(FIDS_TARGET)

SOURCE = Path("data/FIDS30")
for cls_dir in SOURCE.iterdir():
    if not cls_dir.is_dir():
        continue
    dest = FIDS_TARGET / cls_dir.name
    dest.mkdir(parents=True, exist_ok=True)
    for img in cls_dir.glob("*"):
        if img.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
            os.symlink(img.absolute(), dest / img.name)

total = sum(1 for _ in FIDS_TARGET.rglob("*") if _.is_symlink())
print(f"FIDS30: {total} images across {len(list(FIDS_TARGET.iterdir()))} classes")

## 3. Sample Visualization

In [ ]:
import matplotlib.pyplot as plt
from torchvision import datasets

uc_ds = datasets.ImageFolder(str(UC_TARGET / "Training"))
fids_ds = datasets.ImageFolder(str(FIDS_TARGET))

# TODO: Create a 2x6 grid (UC Merced top row, FIDS30 bottom row)
fig, axes = plt.subplots(___, ___, figsize=(15, 5.5))

for j in range(6):
    idx = uc_ds.targets.index(j)
    axes[0, j].imshow(uc_ds[idx][0])
    axes[0, j].set_title(uc_ds.classes[j], fontsize=10)
    axes[0, j].axis('off')
axes[0, 0].set_ylabel("UC Merced\n(normal)", fontsize=11)

for j in range(6):
    idx = fids_ds.targets.index(j)
    axes[1, j].imshow(fids_ds[idx][0])
    axes[1, j].set_title(fids_ds.classes[j], fontsize=10)
    axes[1, j].axis('off')
axes[1, 0].set_ylabel("FIDS30\n(OOD)", fontsize=11)

plt.suptitle("In-Distribution (UC Merced) vs OOD (FIDS30)", fontsize=13)
plt.tight_layout()
plt.show()